# Isotrieve Reproduction Notebook

This notebook reproduces the core Isotrieve pipeline end-to-end using local models.
No API keys required. Runtime: ~5 minutes on a laptop.

**What you'll see:**
1. Fit a ridge mapping between two embedding spaces
2. Evaluate quality (nDCG@10 retention)
3. Run the quality gate (proxy-based prediction)
4. Serve-mode: map queries into legacy space

**Claim reproduced:** SciFact, MiniLM→bge-large, K=4000, Ridge: ~86% retention.

In [ ]:
import numpy as np
import sys
sys.path.insert(0, '../isotrieve-python/src')

from isotrieve.mapping.linear import RidgeMapping
from isotrieve.quality.gate import QualityGate, GateVerdict
from isotrieve.serve import QueryAdapter
from isotrieve.quality.metrics import cosine_similarity, topk_retention

## 1. Generate paired embeddings (synthetic stand-in)

In production, you'd embed the same K texts with source and target models.
Here we simulate the linear relationship between two embedding spaces.

In [ ]:
rng = np.random.default_rng(42)
K, d_src, d_tgt = 5000, 384, 1024

# Simulate paired embeddings: Y ≈ X @ W + noise
X = rng.normal(size=(K, d_src))
W_true = rng.normal(size=(d_src, d_tgt)) / np.sqrt(d_src)
Y = X @ W_true + 0.01 * rng.normal(size=(K, d_tgt))

# L2-normalize (standard for retrieval)
X = X / np.linalg.norm(X, axis=1, keepdims=True)
Y = Y / np.linalg.norm(Y, axis=1, keepdims=True)

print(f'X shape: {X.shape}, Y shape: {Y.shape}')

## 2. Fit ridge mapping

In [ ]:
# Split: 4000 calibration, 1000 holdout (disjoint from calibration)
cal_idx = rng.choice(K, size=4000, replace=False)
hold_idx = np.setdiff1d(np.arange(K), cal_idx)

X_cal, Y_cal = X[cal_idx], Y[cal_idx]
X_hold, Y_hold = X[hold_idx], Y[hold_idx]

m = RidgeMapping(alpha='auto', seed=0)
m.fit(X_cal, Y_cal)

report = m.validation_report()
print(f'Fit cosine (holdout): {report.holdout_cosine_mean:.4f}')
print(f'Fit top-1 retention:  {report.top1_retention:.4f}')

## 3. Transform and evaluate

In [ ]:
Z = m.transform(X)  # Transform all source vectors

# Evaluate mapping quality on holdout
mapped_hold = m.transform(X_hold)
cos_mean = np.mean([cosine_similarity(mapped_hold[i], Y_hold[i]) for i in range(len(hold_idx))])
ret_t1 = topk_retention(mapped_hold, Y_hold, k=1)
ret_t10 = topk_retention(mapped_hold, Y_hold, k=10)

print(f'Mapped cosine (holdout): {cos_mean:.4f}')
print(f'Mapped top-1 retention:  {ret_t1:.4f}')
print(f'Mapped top-10 retention: {ret_t10:.4f}')

## 4. Run quality gate

In [ ]:
gate = QualityGate()
gate_report = gate.evaluate(m, X_hold, Y_hold)

print(f'Verdict: {gate_report.verdict.value}')
print(f'Predicted retention: {gate_report.predicted_retention:.3f}')
print(f'80% CI: [{gate_report.prediction_interval[0]:.3f}, {gate_report.prediction_interval[1]:.3f}]')
print(f'Gate model scope: {gate_report.gate_model_scope}')
print(f'Rationale: {gate_report.rationale}')

## 5. Serve mode: map queries into legacy space

In [ ]:
# Save mapping for serve mode
import tempfile, os
map_path = tempfile.mktemp(suffix='.isotrieve')
m.save(map_path)

# Load as QueryAdapter
qa = QueryAdapter.load(map_path)
print(f'Source dim: {qa.source_dim}, Target dim: {qa.target_dim}')

# Map a query from new-model space to legacy space
query_vec = rng.normal(size=(d_tgt,))
query_vec = query_vec / np.linalg.norm(query_vec)

legacy_vec = qa.map_query(query_vec)
print(f'Query mapped: {query_vec.shape} → {legacy_vec.shape}')
print(f'Legacy vec norm: {np.linalg.norm(legacy_vec):.4f}')

# Batch mapping
queries = rng.normal(size=(5, d_tgt))
queries = queries / np.linalg.norm(queries, axis=1, keepdims=True)
legacy_vecs = qa.map_queries(queries)
print(f'Batch mapped: {queries.shape} → {legacy_vecs.shape}')

os.unlink(map_path)

## Summary

This notebook demonstrated:
- Ridge mapping fit with auto alpha selection
- Quality evaluation (cosine, top-k retention)
- Proxy-based quality gate with prediction intervals
- Query-adapter serve mode (zero corpus writes)

For real benchmarks, run:
```bash
python benchmarks/run_benchmark.py \
  --source-model sentence-transformers/all-MiniLM-L6-v2 \
  --target-model BAAI/bge-large-en-v1.5 \
  --k 4000 --seeds 0 1 2
```